In [1]:
import json
import csv
import math
import time
import warnings
from pathlib import Path
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
import numpy as np
from kiwipiepy import Kiwi

DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/vector_db_hybrid_rrf_morpheme_k"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
EMBEDDING_MODEL = "bge-m3"
BATCH_SIZE      = 500
TOP_K           = 10
CHUNK_SIZE      = 500
OVERLAP         = 100
MERGE_TABLE     = True
BM25_ALPHA      = 0.5
RRF_K           = 60
TOP_K_CANDIDATE = 100

In [15]:
STRATEGIES = [
    {"name": "A_rrf_k10",  "search": "rrf", "rrf_k": 10},
    {"name": "B_rrf_k30",  "search": "rrf", "rrf_k": 30},
    {"name": "C_rrf_k60",  "search": "rrf", "rrf_k": 60},
    {"name": "D_rrf_k80",  "search": "rrf", "rrf_k": 80},
    {"name": "E_rrf_k100", "search": "rrf", "rrf_k": 100},
]

GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    # 다중 문서
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    # 평가 제외
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}

In [3]:
# 형태소 분석
kiwi = Kiwi()

def tokenize(text: str) -> list[str]:
    return [token.form for token in kiwi.tokenize(text)]

def build_bm25(all_texts: list[str]) -> BM25Okapi:
    tokenized = [tokenize(text) for text in all_texts]
    return BM25Okapi(tokenized)

In [4]:
def search_rrf(query, vs, bm25, text_to_idx, k, rrf_k=60):
    vec_hits = vs.similarity_search_with_relevance_scores(query, k=TOP_K_CANDIDATE)

    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_top_indices = np.argsort(-bm25_scores)[:TOP_K_CANDIDATE]
    bm25_rank = {idx: rank+1 for rank, idx in enumerate(bm25_top_indices)}

    combined = []
    for vec_rank, (doc, _) in enumerate(vec_hits):
        idx = text_to_idx.get(doc.page_content)
        b_rank = bm25_rank.get(idx, TOP_K_CANDIDATE + 1) if idx is not None else TOP_K_CANDIDATE + 1
        rrf_score = 1/(rrf_k + vec_rank+1) + 1/(rrf_k + b_rank)
        combined.append((doc, rrf_score))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:k]

In [5]:
def clean(val):
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [6]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }


In [7]:
def chunk_section(section: dict) -> list[dict]:
    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue

        content = block["content"]

        if block.get("needs_subsplit") and len(content) > CHUNK_SIZE:
            if current_text.strip():
                results.append({"content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}", "block": last_text_block})
                current_text = current_text[-OVERLAP:] if OVERLAP else ""
                last_text_block = None
            i = 0
            while i < len(content):
                sub = content[i:i+CHUNK_SIZE]
                next_i = i + CHUNK_SIZE - OVERLAP
                is_last = next_i >= len(content)
                if is_last and len(sub) < OVERLAP * 2:
                    current_text = sub + "\n\n"
                    last_text_block = block
                else:
                    results.append({"content": f"[섹션: {header_prefix}]\n\n{sub}", "block": block})
                i = next_i
            continue

        if block["type"] == "table":
            combined = (current_text + "\n\n" + content).strip()
            if len(combined) > CHUNK_SIZE and current_text.strip():
                results.append({"content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}", "block": last_text_block})
                current_text = current_text[-OVERLAP:] if OVERLAP else ""
                results.append({"content": f"[섹션: {header_prefix}]\n\n{content}", "block": block})
                current_text = ""
                last_text_block = None
            else:
                current_text = combined + "\n\n"
                last_text_block = block
        else:
            if len(current_text) + len(content) > CHUNK_SIZE and current_text.strip():
                results.append({"content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}", "block": last_text_block})
                prev_text = current_text[-OVERLAP:] if OVERLAP else ""
                current_text = prev_text + content + "\n\n"
            else:
                current_text += content + "\n\n"
            last_text_block = block

    if current_text.strip():
        results.append({"content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}", "block": last_text_block})

    return results

In [8]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    texts, metas = [], []
    meta = doc.get("metadata", {})
    name = str(clean(meta.get("사업명")))
    org  = str(clean(meta.get("발주기관")))
    prefix = f"[사업명] {name}\n[발주기관] {org}\n\n"

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

In [9]:
def load_embedding_model(name: str):
    if name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "text-embedding-3-small":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

In [10]:
def save_vectorstore(strategy: dict, all_texts: list, all_metas: list, embedding_model):
    chroma_dir = f"{CHROMA_BASE}/{strategy['name']}"

    if Path(chroma_dir).exists():
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        existing_count = vs._collection.count()
        if existing_count >= len(all_texts):
            print(f"[SKIP] {strategy['name']} — 완료된 DB 존재 ({existing_count}개 청크)")
            return vs
        print(f"[RESUME] {strategy['name']} — {existing_count}/{len(all_texts)}개, 이어서 진행")
        start_from = existing_count
    else:
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        start_from = 0

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vs.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vs._collection.count()}개 청크) → {chroma_dir}")
    return vs

In [11]:
def hit(retrieved_ids, target_ids, k):
    return any(tid in retrieved_ids[:k] for tid in target_ids)

In [12]:
# 골든셋 로드
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

json_files = sorted(Path(DOCS_DIR).glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

embedding_model = load_embedding_model(EMBEDDING_MODEL)
warnings.filterwarnings("ignore", message="Relevance scores must be between")

골든셋 문항 수: 101
JSON 파일 수: 98


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [13]:
eval_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*50}")
    print(f"[전략] {strategy['name']} (rrf_k={strategy['rrf_k']})")

    all_texts, all_metas = [], []
    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        texts, metas = process_doc(doc)
        all_texts.extend(texts)
        all_metas.extend(metas)

    print(f"  총 청크: {len(all_texts)}개 | 평균 길이: {sum(len(t) for t in all_texts)//len(all_texts)}자")

    vs = save_vectorstore(strategy, all_texts, all_metas, embedding_model)
    bm25 = build_bm25(all_texts)
    text_to_idx = {text: i for i, text in enumerate(all_texts)}

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)
        if target is None:
            skipped += 1
            continue
        target_ids = target if isinstance(target, list) else [target]

        t0 = time.time()
        hits = search_rrf(row["question"], vs, bm25, text_to_idx, TOP_K, rrf_k=strategy["rrf_k"])
        query_times.append(time.time() - t0)

        retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]

        recall3.append(1.0 if hit(retrieved_doc_ids, target_ids, 3) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, target_ids, 5) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, target_ids, 10) else 0.0)
        rank = next((i+1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids), None)
        mrr_scores.append(1.0/rank if rank else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")
    eval_results.append({
        "전략":      strategy["name"],
        "rrf_k":     strategy["rrf_k"],
        "Recall@3":  round(sum(recall3)/n, 4),
        "Recall@5":  round(sum(recall5)/n, 4),
        "Recall@10": round(sum(recall10)/n, 4),
        "MRR":       round(sum(mrr_scores)/n, 4),
        "avg_ms":    round(sum(query_times)/n*1000, 1),
        "청크수":    len(all_texts),
    })


[전략] A_rrf_k10 (rrf_k=10)
  총 청크: 16190개 | 평균 길이: 518자
  저장 완료: 500/16190
  저장 완료: 1000/16190
  저장 완료: 1500/16190
  저장 완료: 2000/16190
  저장 완료: 2500/16190
  저장 완료: 3000/16190
  저장 완료: 3500/16190
  저장 완료: 4000/16190
  저장 완료: 4500/16190
  저장 완료: 5000/16190
  저장 완료: 5500/16190
  저장 완료: 6000/16190
  저장 완료: 6500/16190
  저장 완료: 7000/16190
  저장 완료: 7500/16190
  저장 완료: 8000/16190
  저장 완료: 8500/16190
  저장 완료: 9000/16190
  저장 완료: 9500/16190
  저장 완료: 10000/16190
  저장 완료: 10500/16190
  저장 완료: 11000/16190
  저장 완료: 11500/16190
  저장 완료: 12000/16190
  저장 완료: 12500/16190
  저장 완료: 13000/16190
  저장 완료: 13500/16190
  저장 완료: 14000/16190
  저장 완료: 14500/16190
  저장 완료: 15000/16190
  저장 완료: 15500/16190
  저장 완료: 16000/16190
  저장 완료: 16190/16190
  완료 (16190개 청크) → ../../data/vector_db_hybrid_rrf_morpheme_k/A_rrf_k10
  평가: 84개 | 제외: 17개

[전략] B_rrf_k30 (rrf_k=30)
  총 청크: 16190개 | 평균 길이: 518자
  저장 완료: 500/16190
  저장 완료: 1000/16190
  저장 완료: 1500/16190
  저장 완료: 2000/16190
  저장 완료: 2500/16190
  저장 완료: 3000/16190
  저장

In [16]:
from IPython.display import Markdown, display

header = "| 전략 | rrf_k | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |\n|------|-------|----------|----------|-----------|-----|--------|"
rows = "\n".join(
    f"| {r['전략']} | {r['rrf_k']} | {r['Recall@3']} | {r['Recall@5']} | {r['Recall@10']} | {r['MRR']} | {r['avg_ms']} |"
    for r in eval_results
)
display(Markdown(f"## RRF K값 실험 결과\n\n{header}\n{rows}"))

## RRF K값 실험 결과

| 전략 | rrf_k | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |
|------|-------|----------|----------|-----------|-----|--------|
| A_rrf_k10 | 10 | 0.2738 | 0.2857 | 0.3452 | 0.2293 | 94.6 |
| B_rrf_k30 | 30 | 0.2619 | 0.2738 | 0.3452 | 0.2269 | 101.5 |
| C_rrf_k50 | 60 | 0.2619 | 0.2976 | 0.369 | 0.2515 | 100.2 |
| D_rrf_k70 | 80 | 0.2619 | 0.2976 | 0.369 | 0.2227 | 102.4 |
| E_rrf_k100 | 100 | 0.2619 | 0.2976 | 0.369 | 0.2375 | 101.5 |